# Basic Probability
**Topic:** Probability & Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Define** what probability means as a number between 0 and 1
- **Calculate** simple probabilities by counting favorable and total outcomes
- **Interpret** complement probability and why P(A) + P(not A) always equals 1

> **Tip:** Start by moving the **number of favorable outcomes slider** and watch how the probability bar and complement update instantly.

---
## How we got here

This is the first notebook in the series. Probability is the foundation everything else rests on, before we can talk about distributions, confidence intervals, or hypothesis tests, we need a shared language for measuring uncertainty. This notebook establishes that language.

---
## Why this matters for data science

Every machine learning model that outputs a score, a spam classifier, a fraud detector, a click-through predictor, is really outputting a probability. Understanding what that number means, and how it relates to outcomes and errors, is the prerequisite for evaluating any model you will ever build.

---
## Try it yourself

In [2]:
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import VBox, HBox
from IPython.display import display
from tkh_utils import PALETTE, FONT, base_layout

# 1. Create controls
favorable_slider = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description="Favorable:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px"),
)
total_slider = widgets.IntSlider(
    value=10, min=1, max=20, step=1,
    description="Total outcomes:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px"),
)
experiment_dropdown = widgets.Dropdown(
    options=[
        ("Coin flip", "coin"),
        ("Dice roll", "dice"),
        ("Card draw", "card"),
        ("Custom (use sliders)", "custom"),
    ],
    value="custom",
    description="Experiment:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px"),
)

# 2. Create FigureWidget ONCE
fig = go.FigureWidget(
    data=[
        go.Bar(
            x=["Favorable", "Unfavorable"],
            y=[3, 7],
            marker_color=[PALETTE["primary"], PALETTE["secondary"]],
            text=[3, 7],
            textposition="outside",
        )
    ],
    layout=base_layout(
        title="Outcomes: 3 of 10 are favorable",
        xaxis_title="Outcome type",
        yaxis_title="Count",
    ),
)
fig.update_layout(showlegend=False, yaxis=dict(range=[0, 18]), height=360)

# 3. Create output label widget
prob_label = widgets.HTML(value="<h3>P(A) = 3 / 10</h3>")

# 4. Update function — modifies FigureWidget IN PLACE, never creates new figure
def update(change=None):
    exp = experiment_dropdown.value

    if exp == "coin":
        fav, total = 1, 2
    elif exp == "dice":
        fav, total = 1, 6
    elif exp == "card":
        fav, total = 13, 52
    else:
        fav   = favorable_slider.value
        total = total_slider.value

    fav   = min(fav, total)
    unfav = total - fav
    prob  = fav / total

    is_preset = exp in ("coin", "dice", "card")
    favorable_slider.disabled = is_preset
    total_slider.disabled     = is_preset

    with fig.batch_update():
        fig.data[0].y          = [fav, unfav]
        fig.data[0].text       = [fav, unfav]
        fig.layout.title.text  = f"Outcomes: {fav} of {total} are favorable"
        fig.layout.yaxis.range = [0, max(total, fav + 1) * 1.35]

    # Pre-extract palette colors — nested quotes in f-strings require Python 3.12+
    col_primary   = PALETTE["primary"]
    col_secondary = PALETTE["secondary"]
    prob_label.value = (
        f"<div style='font-family:Inter,Arial,sans-serif; padding:14px 0; line-height:2.2;'>"
        f"<span style='font-size:22px; font-weight:700; color:{col_primary};'>"
        f"P(A) &nbsp;=&nbsp; {fav} / {total}</span><br>"
        f"<span style='font-size:15px; color:#333;'>"
        f"&nbsp;&nbsp;&nbsp;= &nbsp;{prob:.4f}"
        f"&nbsp;&nbsp;=&nbsp; {prob * 100:.2f}%</span><br>"
        f"<span style='font-size:15px; color:{col_secondary};'>"
        f"P(not A) &nbsp;=&nbsp; 1 &minus; {prob:.4f}"
        f" &nbsp;=&nbsp; {1 - prob:.4f}"
        f" &nbsp;=&nbsp; {(1 - prob) * 100:.2f}%</span>"
        f"</div>"
    )

# 5. Register observers
favorable_slider.observe(update, names="value")
total_slider.observe(update, names="value")
experiment_dropdown.observe(update, names="value")

# 6. Display ONCE — nothing else after this
controls = VBox(
    [experiment_dropdown, favorable_slider, total_slider, prob_label],
    layout=widgets.Layout(width="420px", padding="0 24px 0 0"),
)
display(HBox([controls, fig]))

# 7. Trigger initial render
update()


---
## What's happening?

Probability is a way of assigning a number to how likely an event is to occur. The simplest version, called **classical probability**, assumes every outcome is equally likely, so we just count.

| Symbol | What it controls |
|--------|-----------------|
| n | Total number of equally likely outcomes |
| k | Number of outcomes where event A occurs |
| P(A) | Probability that A happens |

$$P(A) = \frac{k}{n}$$

### The complement rule

For any event A, the probability it does **not** happen is exactly 1 minus the probability it does:

$$P(\text{not } A) = 1 - P(A)$$

Go back to the widget and drag the favorable outcomes slider all the way to the right, P(not A) collapses to zero, while P(A) reaches 1. They always sum to exactly 1 because one of the two outcomes must happen.

---
## Real-world example: Free-throw shooting in basketball

A basketball player's free-throw percentage is a classic probability in action. Each shot is an independent trial with two outcomes, made or missed, and the historical rate gives us our best estimate of P(make).

The chart below shows the free-throw percentages for a sample of NBA players from the 2022–23 season. Notice:

- **Notice:** Most players cluster between 70% and 85%, forming a rough bell shape rather than a uniform spread
- **Notice:** No player reaches 100%, even the best shooters miss sometimes, which is exactly what P(A) < 1 means
- **Notice:** The complement (miss rate) is simply the gap between each bar and 1.0, a player shooting 80% misses 20% of the time

> **Discussion question:** If a player has made 73 of their last 100 free throws, is P(make) = 0.73 a fact or an estimate? What would you need to assume to treat it as a true probability?

In [3]:
np.random.seed(7)

# Realistic NBA free-throw percentages (2022-23 season approximation)
players = [
    "S. Curry", "D. Nowitzki*", "K. Durant", "D. DeRozan", "J. Harden",
    "L. James", "G. Antetokounmpo", "J. Embiid", "T. Young", "D. Mitchell",
    "B. Ingram", "Z. LaVine", "C. Anthony", "P. George", "K. Irving",
]
pct = [0.916, 0.878, 0.872, 0.862, 0.857,
       0.731, 0.644, 0.857, 0.836, 0.832,
       0.819, 0.845, 0.821, 0.876, 0.900]

sorted_pairs = sorted(zip(pct, players), reverse=True)
pct_s, players_s = zip(*sorted_pairs)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=players_s, y=pct_s,
    marker_color=PALETTE["primary"],
    name="P(make)",
))
fig.add_trace(go.Bar(
    x=players_s, y=[1 - p for p in pct_s],
    marker_color=PALETTE["secondary"],
    name="P(miss) = complement",
    opacity=0.7,
))
layout = base_layout(
    title="Free-Throw Probability — NBA Sample (2022–23)",
    xaxis_title="Player",
    yaxis_title="Probability",
)
layout.update(barmode="stack", yaxis=dict(range=[0, 1.05], tickformat=".0%"))
fig.update_layout(layout)
fig.show()

### Other everyday probability examples

| Situation | Event A | P(A) estimate | Complement P(not A) |
|-----------|---------|--------------|---------------------|
| Coin flip | Heads | 0.50 | 0.50 |
| Six-sided die | Roll a 4 | 0.167 | 0.833 |
| Drawing a heart from a deck | Heart card | 0.25 | 0.75 |
| Rain on a random July day in Miami | Rain | ~0.60 | ~0.40 |
| Email is spam (typical inbox) | Spam | ~0.45 | ~0.55 |

---
## Key takeaway

> **Probability measures how often an event occurs out of all possible outcomes; its complement fills the remaining space — and the two always sum to exactly 1.**

---
*Next up: Conditional Probability & Independence — what happens to P(A) when you already know something else is true*